# Support Vector Machine — Philippine Employee Stagnation

Third model in the comparison pipeline. SVM finds the widest possible decision boundary between stagnant and non-stagnant workers by maximising the margin between the two classes.

**Contents:**
1. Load processed data
2. Tune C (regularisation) and kernel via cross-validation
3. Train final SVM model
4. Classification report
5. Precision-Recall and ROC curves
6. Full three-model comparison table

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils import resample
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, ConfusionMatrixDisplay,
)
import joblib

BASE = os.path.join(os.path.dirname(os.getcwd()), 'ph-lfs-stagnation - Copy - Copy - Copy')
if not os.path.exists(os.path.join(BASE, 'data')):
    BASE = os.path.dirname(os.getcwd())
if not os.path.exists(os.path.join(BASE, 'data')):
    BASE = os.getcwd()

DATA_DIR   = os.path.join(BASE, 'data')
MODELS_DIR = os.path.join(BASE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print('BASE:', BASE)

## 1. Load Data

In [ ]:
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv')).squeeze()
y_test  = pd.read_csv(os.path.join(DATA_DIR, 'y_test.csv')).squeeze()

FEATURE_COLS = X_train.columns.tolist()
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train stagnation rate: {y_train.mean()*100:.2f}%')
print(f'Test  stagnation rate: {y_test.mean()*100:.2f}%')

## 2. Tune C and Kernel via Cross-Validation

SVC with `probability=True` uses Platt scaling (logistic calibration on top of the SVM output) to produce class probabilities needed for AUC-PR.

**Sampling note:** Full SVC training is O(n²–n³). We tune on a 50k stratified subsample and train the final model on the full dataset.

In [ ]:
TUNE_SAMPLE = 50_000
idx = resample(range(len(X_train)), n_samples=min(TUNE_SAMPLE, len(X_train)),
               stratify=y_train, random_state=42)
X_tune = X_train.iloc[idx]
y_tune = y_train.iloc[idx]
print(f'Tuning sample: {len(X_tune):,} rows  (stagnation rate: {y_tune.mean()*100:.2f}%)')

scaler  = StandardScaler()
X_tune_sc = scaler.fit_transform(X_tune)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Grid: C values x kernels
grid = [
    ('rbf',    0.1),
    ('rbf',    1.0),
    ('rbf',   10.0),
    ('linear', 0.1),
    ('linear', 1.0),
]

results = []
for kernel, C in grid:
    svc = SVC(kernel=kernel, C=C, class_weight='balanced',
              probability=True, random_state=42)
    scores = cross_val_score(svc, X_tune_sc, y_tune, cv=cv,
                              scoring='average_precision', n_jobs=-1)
    results.append((kernel, C, scores.mean(), scores.std()))
    print(f'  kernel={kernel:<7} C={C:<5} AUC-PR={scores.mean():.4f} +/- {scores.std():.4f}')

best = max(results, key=lambda x: x[2])
best_kernel, best_C = best[0], best[1]
print(f'\nBest: kernel={best_kernel}  C={best_C}  (AUC-PR={best[2]:.4f})')

## 3. Train Final SVM Model

`class_weight='balanced'` adjusts the soft-margin penalty inversely proportional to class frequency — stagnant misclassifications are penalised more heavily.

In [ ]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(
        kernel=best_kernel,
        C=best_C,
        class_weight='balanced',
        probability=True,
        random_state=42,
    ))
])

print('Fitting SVM on full training set (may take a few minutes)...')
pipe_svm.fit(X_train, y_train)
print('Done.')
joblib.dump(pipe_svm, os.path.join(MODELS_DIR, 'svm.pkl'))
print('Saved to models/svm.pkl')

## 4. Classification Report

In [ ]:
y_prob_svm = pipe_svm.predict_proba(X_test)[:, 1]
y_pred_svm = pipe_svm.predict(X_test)

print('=' * 55)
print('SVM CLASSIFICATION REPORT  (threshold = 0.5)')
print('=' * 55)
print(classification_report(y_test, y_pred_svm,
                             target_names=['Not Stagnant', 'Stagnant'], digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_svm),
                       display_labels=['Not Stagnant', 'Stagnant']).plot(
    ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('SVM — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'svm_confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. Precision-Recall and ROC Curves

In [ ]:
ap_svm  = average_precision_score(y_test, y_prob_svm)
roc_svm = roc_auc_score(y_test, y_prob_svm)

prec_s, rec_s, _ = precision_recall_curve(y_test, y_prob_svm)
fpr_s,  tpr_s, _ = roc_curve(y_test, y_prob_svm)

# Load other models for comparison
lr_pipe  = joblib.load(os.path.join(MODELS_DIR, 'logistic_regression.pkl'))
knn_pipe = joblib.load(os.path.join(MODELS_DIR, 'knn.pkl'))
y_prob_lr  = lr_pipe.predict_proba(X_test)[:, 1]
y_prob_knn = knn_pipe.predict_proba(X_test)[:, 1]

ap_lr   = average_precision_score(y_test, y_prob_lr)
ap_knn  = average_precision_score(y_test, y_prob_knn)
roc_lr  = roc_auc_score(y_test, y_prob_lr)
roc_knn = roc_auc_score(y_test, y_prob_knn)

prec_l, rec_l, _ = precision_recall_curve(y_test, y_prob_lr)
prec_k, rec_k, _ = precision_recall_curve(y_test, y_prob_knn)
fpr_l,  tpr_l, _ = roc_curve(y_test, y_prob_lr)
fpr_k,  tpr_k, _ = roc_curve(y_test, y_prob_knn)

COLOURS = {'LR': '#DD8452', 'KNN': '#4C72B0', 'SVM': '#55A868'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(rec_l, prec_l, color=COLOURS['LR'],  linewidth=2,   linestyle='--', label=f'Logistic Reg (AUC-PR={ap_lr:.4f})')
ax.plot(rec_k, prec_k, color=COLOURS['KNN'], linewidth=2,   linestyle=':',  label=f'KNN          (AUC-PR={ap_knn:.4f})')
ax.plot(rec_s, prec_s, color=COLOURS['SVM'], linewidth=2.5,               label=f'SVM          (AUC-PR={ap_svm:.4f})')
ax.axhline(y_test.mean(), color='grey', linestyle=':', linewidth=1.5, label='Random baseline')
ax.set_xlim([0,1]); ax.set_ylim([0,1])
ax.set_xlabel('Recall', fontsize=12); ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — All 3 Models', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(fpr_l, tpr_l, color=COLOURS['LR'],  linewidth=2,   linestyle='--', label=f'Logistic Reg (AUC-ROC={roc_lr:.4f})')
ax.plot(fpr_k, tpr_k, color=COLOURS['KNN'], linewidth=2,   linestyle=':',  label=f'KNN          (AUC-ROC={roc_knn:.4f})')
ax.plot(fpr_s, tpr_s, color=COLOURS['SVM'], linewidth=2.5,               label=f'SVM          (AUC-ROC={roc_svm:.4f})')
ax.plot([0,1], [0,1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — All 3 Models', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('Three-Model Comparison — Test Set (2024)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'three_model_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

## 6. Full Three-Model Comparison Table

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def optimal_threshold_f1(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best = np.argmax(f1)
    return thr[best], f1[best]

rows = []
for name, y_prob in [('Logistic Regression', y_prob_lr),
                      ('KNN',                 y_prob_knn),
                      ('SVM',                 y_prob_svm)]:
    ap   = average_precision_score(y_test, y_prob)
    roc  = roc_auc_score(y_test, y_prob)
    thr, best_f1 = optimal_threshold_f1(y_test, y_prob)
    y_pred_opt = (y_prob >= thr).astype(int)
    p = precision_score(y_test, y_pred_opt, zero_division=0)
    r = recall_score(y_test, y_pred_opt, zero_division=0)
    rows.append({'Model': name, 'AUC-PR': ap, 'AUC-ROC': roc,
                 'Opt Threshold': thr, 'F1 (stagnant)': best_f1,
                 'Precision': p, 'Recall': r})

comp = pd.DataFrame(rows).set_index('Model')
print('=' * 75)
print('THREE-MODEL COMPARISON — Test Set (2024)')
print('=' * 75)
print(comp.to_string(float_format='{:.4f}'.format))

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(comp))
w = 0.35
ax.bar(x - w/2, comp['AUC-PR'],  width=w, label='AUC-PR',  color='#4C72B0')
ax.bar(x + w/2, comp['AUC-ROC'], width=w, label='AUC-ROC', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(comp.index, fontsize=12)
ax.set_ylim([0, 1])
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — AUC-PR and AUC-ROC', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'model_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

best_model = comp['AUC-PR'].idxmax()
print(f'\nBest model by AUC-PR: {best_model}  ({comp.loc[best_model, "AUC-PR"]:.4f})')